In [71]:
import pandas as pd
from datetime import datetime
import numpy as np
from sqlalchemy import create_engine



In [72]:
csvsales="../Data/sales2.csv"

In [73]:
mysql_user="root"
mysql_password="atharv30"
mysql_host="localhost"
mysql_port="3306"
mysql_db="sales_db"
target_table="sales_data"

In [74]:
#sqlalchemy connection string
conn=(f"mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{mysql_db}")

In [75]:
engine=create_engine(conn)

In [76]:
try:
    df=pd.read_csv(csvsales)
    print("CSV file read successfully.")
    shape=df.shape
    print(f"DataFrame shape: {shape}")
    display(df.head())
except Exception as e:
    print(f"Error reading CSV file: {e}")


CSV file read successfully.
DataFrame shape: (10, 7)


,OrderId,Product,Category,SalesAmount,OrderDate,Region,CustomerName
0,1011,Monitor Stand,Furniture,55.00,2025-01-11,West,Kyle Reese
1,1012,Noise Cancelling Buds,Electronics,180.00,2025-01-11,East,Laura Croft
2,1013,Air Fryer Large,Home Goods,110.00,2025-01-12,Central,Michael Scott
3,1014,Ergonomic Mousepad,Electronics,15.99,2025-01-12,South,Nancy Drew
4,1015,4-Port USB Hub,Electronics,35.00,2025-01-13,East,Oliver Queen


In [77]:
df.columns=(
    df.columns
     .str.replace(' ', '_')
     .str.replace(r'([A-Z])',r'_\1',regex=True)
     .str.lower()
     .str.strip('_')
)

In [78]:
df.columns


Index(['order_id', 'product', 'category', 'sales_amount', 'order_date',
       'region', 'customer_name'],
      dtype='str')

In [79]:
df.dtypes

order_id           int64
product              str
category             str
sales_amount     float64
order_date           str
region               str
customer_name        str
dtype: object

In [80]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

In [81]:
df['sales_amount'] = pd.to_numeric(df['sales_amount'], errors='coerce')

In [82]:
df['unity_price']=df['sales_amount']



In [83]:
condition=[
    df['sales_amount'] >= 500,
    df['sales_amount'] >= 100,
]
choices=['high','medium']
df['sales_tier']=np.select(condition,choices,default='low')

In [84]:
df=df[df['sales_amount']>0]

In [85]:
df['load_timestamp']=pd.to_datetime(datetime.now())

In [86]:
df

,order_id,product,category,sales_amount,order_date,region,customer_name,unity_price,sales_tier,load_timestamp
0,1011,Monitor Stand,Furniture,55.00,2025-01-11,West,Kyle Reese,55.00,low,2026-06-18 00:24:35.421578
1,1012,Noise Cancelling Buds,Electronics,180.00,2025-01-11,East,Laura Croft,180.00,medium,2026-06-18 00:24:35.421578
2,1013,Air Fryer Large,Home Goods,110.00,2025-01-12,Central,Michael Scott,110.00,medium,2026-06-18 00:24:35.421578
3,1014,Ergonomic Mousepad,Electronics,15.99,2025-01-12,South,Nancy Drew,15.99,low,2026-06-18 00:24:35.421578
4,1015,4-Port USB Hub,Electronics,35.00,2025-01-13,East,Oliver Queen,35.00,low,2026-06-18 00:24:35.421578
5,1016,Gaming Mouse Pro,Electronics,85.00,2025-01-13,West,Pam Beesly,85.00,low,2026-06-18 00:24:35.421578
6,1017,Smart Watch Series 5,Electronics,299.99,2025-01-14,Central,Quentin Tarantino,299.99,medium,2026-06-18 00:24:35.421578
7,1018,Small Blender,Home Goods,49.99,2025-01-14,South,Rachel Green,49.99,low,2026-06-18 00:24:35.421578
8,1019,Desk Organizer,Furniture,25.00,2025-01-15,East,Steve Rogers,25.00,low,2026-06-18 00:24:35.421578
9,1020,3D Printer Starter,Electronics,950.00,2025-01-15,West,Tony Stark,950.00,high,2026-06-18 00:24:35.421578


In [87]:
#load into mysql
try:
    df.to_sql(
        name=target_table, 
        con=engine, 
        if_exists='append', 
        index=False,
        chunksize=1000)
    print("Data loaded into MySQL successfully.")
except Exception as e:
    print(f"Error loading data into MySQL: {e}")

Data loaded into MySQL successfully.
